# 07 · Skor geopolitik artikel × sentimen (eksploratif, MLP tanpa tuning)
Skor artikel `g` = jumlah kata kamus geopolitik / total kata artikel (kamus biner, `src/geo_terms.txt`).
Dua cara memakai `g` untuk tiap sumber NLP: **g × skor** (sentimen diperkuat) dan **rata-rata berbobot g** per hari (artikel lebih geopolitik lebih berpengaruh; untuk embedding dan probabilitas).
Pakai cache yang sudah ada, tanpa ekstraksi ulang. MLP: hyperparameter tetap sama dengan 05, 5 seed.

In [1]:
import sys; sys.path.insert(0, "../src")
import re, json
from collections import Counter
import numpy as np, pandas as pd
from sklearn.neural_network import MLPRegressor
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

X = pd.read_parquet(OUT / "features.parquet")
blocks = json.loads((OUT / "blocks.json").read_text())
prices, articles, sent = t3.load_export(); days = sent.index
date = articles.aligned_trading_date.values

## Skor geopolitik per artikel

In [2]:
terms = [t.strip().lower() for t in (t3.ROOT / "src/geo_terms.txt").read_text().splitlines() if t.strip() and not t.startswith("#")]
pat = re.compile(r"\b(?:" + "|".join(re.escape(t) for t in sorted(set(terms), key=len, reverse=True)) + r")(?:s|es|ed|ing)?\b")
text = (articles.title + " " + articles.full_text).str.lower()
found = text.map(pat.findall)
g = found.map(len) / text.map(lambda s: len(s.split()))
print(f"{len(set(terms))} istilah | artikel tanpa kata kamus: {(g == 0).mean():.1%} | g mean {g.mean():.4f}, median {g.median():.4f}, max {g.max():.4f}")
print("Kata kamus paling sering:", Counter(m for f in found for m in f).most_common(15))

99 istilah | artikel tanpa kata kamus: 0.3% | g mean 0.0204, median 0.0166, max 0.0896
Kata kamus paling sering: [('war', 7190), ('military', 4742), ('conflict', 3913), ('geopolitical', 2770), ('defense', 2684), ('strait', 2220), ('tariffs', 2144), ('nuclear', 2027), ('tensions', 1976), ('attack', 1735), ('sanctions', 1712), ('nato', 1620), ('invasion', 1483), ('attacks', 1459), ('crisis', 1267)]


## Fitur harian: plain vs geo-weighted per sumber NLP

In [3]:
wsum, cnt = g.groupby(date).transform("sum"), g.groupby(date).transform("size")
wn = pd.Series(np.where(wsum > 0, g / wsum.where(wsum > 0, 1), 1 / cnt), index=articles.index)   # bobot per hari, fallback seragam

def rows(df): return df.reindex(days).fillna(0.0).reindex(prices.date).set_axis(prices.index)
def wmean(V): return rows(V.mul(wn, axis=0).groupby(date).sum())      # rata-rata berbobot g
def prod(V): return rows(V.mul(g, axis=0).groupby(date).mean())       # g x skor
geo_mean = rows(g.groupby(date).mean().to_frame("geo_mean"))

SRC = {}   # nama -> (plain, geo)
sa = pd.read_parquet(t3.EXPORT / "sentiment_articles.parquet").sort_values("article_id").reset_index(drop=True)
assert (sa.article_id.values == articles.article_id.values).all()
SRC["sent_lama"] = (X[blocks["sent_lama"]], prod(sa.drop(columns=["article_id", "aligned_trading_date"]).fillna(0.0)).add_prefix("sent_lama_gs_"))

def probs_src(name, P):
    plain = t3.daily_finbert(articles, P, days).reindex(prices.date).set_axis(prices.index)[["fb_mean", "fb_pos", "fb_neg", "fb_neu"]].add_prefix(f"{name}_")
    geo = pd.concat([wmean(P).add_prefix("gw_"), prod((P.pos - P.neg).to_frame("score")).add_prefix("gs_")], axis=1).add_prefix(f"{name}_")
    return plain, geo

SRC["finbert"] = probs_src("finbert", t3.load_finbert(articles))
if (t3.EXPORT / "deberta_ft_probs.npy").exists():
    P = np.load(t3.EXPORT / "deberta_ft_probs.npy"); meta = json.loads((t3.EXPORT / "deberta_ft_probs.json").read_text()); t3._check(meta, articles, len(P))
    short = {"bullish": "pos", "positive": "pos", "bearish": "neg", "negative": "neg", "neutral": "neu"}
    SRC["deberta_ft"] = probs_src("deberta_ft", pd.DataFrame(P, columns=[short[l] for l in meta["labels"]]))
for tag in [*CFG["emb_tags"], "deberta_base"]:
    if (t3.EXPORT / f"embeddings_{tag}.npy").exists():
        E = pd.DataFrame(t3.load_emb(tag, articles)[0])
        SRC[tag] = (rows(E.groupby(date).mean()).add_prefix(f"emb_{tag}_"), wmean(E).add_prefix(f"emb_geo{tag}_"))

base = X[["date", "split", "ret", "target", *PRICE]]
X2 = pd.concat([base, geo_mean, *[f for p, q in SRC.values() for f in (p, q)]], axis=1)
X2 = X2.loc[:, ~X2.columns.duplicated()]
SETS = {"A": PRICE, "geo_only": PRICE + ["geo_mean"]}
for n, (p, q) in SRC.items():
    SETS[f"{n} | plain"] = PRICE + list(p.columns)
    SETS[f"{n} | geo"] = PRICE + list(q.columns) + ["geo_mean"]
tr, va, te = (X2[X2.split == s] for s in ("train", "val", "test"))
print(len(SETS), "set fitur")

14 set fitur


## Korelasi skor (Train): apakah g memperkuat hubungan sentimen dengan return besok?

In [4]:
pairs = {"VADER judul": ("vader_title_mean", "sent_lama_gs_vader_title"), "LM isi net": ("lm_text_net", "sent_lama_gs_lm_text_net"),
         "FinBERT": ("finbert_fb_mean", "finbert_gs_score"), "DeBERTa ft": ("deberta_ft_fb_mean", "deberta_ft_gs_score")}
out = {}
for n, (p, q) in pairs.items():
    for lab, c in (("plain", p), ("g × skor", q)):
        if c in tr: out[(n, lab)] = {"corr ret_t+1": tr[c].corr(tr.target), "corr |ret_t+1|": tr[c].corr(tr.target.abs())}
pd.DataFrame(out).T.round(3)

corr ret_t+1  corr |ret_t+1|
VADER judul plain            0.016           0.001
            g × skor         0.026           0.002
LM isi net  plain           -0.029          -0.038
            g × skor        -0.035          -0.046
FinBERT     plain           -0.017          -0.032
            g × skor        -0.040          -0.045
DeBERTa ft  plain            0.006           0.031
            g × skor        -0.025          -0.009

## MLP: plain vs geo per sumber (Val dan Test)

In [5]:
mlp = lambda seed: MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, max_iter=300, random_state=seed)
res, PT = [], {}
for s, c in SETS.items():
    models = [t3.make_pipe(c, mlp(sd)).fit(tr, tr.target) for sd in range(SEED, SEED + 5)]
    PT[s] = t3.predict(models, te)
    for name, D in (("Val", va), ("Test", te)):
        ev = pd.DataFrame([evaluate(D.target, m.predict(D)) for m in models])
        res.append({"set": s, "split": name, **ev.mean()[["RMSE", "DirAcc", "PredStd"]], "RMSE_std": ev.RMSE.std(ddof=0),
                    "collapse": is_collapsed(t3.predict(models, D), D.target)})
res = pd.DataFrame(res); res.to_csv(OUT / "geo_mlp_results.csv", index=False)
for name, D in (("Val", va), ("Test", te)):
    print(name, "| Prediksi 0 RMSE:", round(evaluate(D.target, np.zeros(len(D)))["RMSE"], 6), "| DirAcc mayoritas:", round(majority_baselines(D.target)["Mayoritas"], 3))
res.pivot_table(index="set", columns="split", values=["RMSE", "DirAcc"], sort=False).round(6)

Val | Prediksi 0 RMSE: 0.003382 | DirAcc mayoritas: 0.534
Test | Prediksi 0 RMSE: 0.002908 | DirAcc mayoritas: 0.591


RMSE              DirAcc          
split                      Val      Test       Val      Test
set                                                         
A                     0.003624  0.003230  0.513636  0.486364
geo_only              0.003686  0.003272  0.488636  0.482955
sent_lama | plain     0.004243  0.003774  0.488636  0.505682
sent_lama | geo       0.004154  0.003780  0.467045  0.477273
finbert | plain       0.003661  0.003337  0.502273  0.487500
finbert | geo         0.003826  0.003400  0.477273  0.472727
deberta_ft | plain    0.003769  0.003216  0.500000  0.520455
deberta_ft | geo      0.003843  0.003281  0.477273  0.506818
bge_small | plain     0.004014  0.003656  0.480682  0.517045
bge_small | geo       0.004181  0.003685  0.470455  0.495455
bge_base | plain      0.004117  0.003821  0.515909  0.471591
bge_base | geo        0.004195  0.003658  0.502273  0.501136
deberta_base | plain  0.003984  0.003585  0.532955  0.520455
deberta_base | geo    0.003878  0.003569  0.526136  0.570455

In [6]:
# geo vs plain per sumber: selisih RMSE (negatif = geo lebih baik) dan p-value DM di Test
r = res.pivot_table(index="set", columns="split", values=["RMSE", "DirAcc"])
cmp = []
for n in SRC:
    a, b = f"{n} | geo", f"{n} | plain"
    cmp.append({"sumber": n, "dRMSE Val": r.loc[a, ("RMSE", "Val")] - r.loc[b, ("RMSE", "Val")], "dRMSE Test": r.loc[a, ("RMSE", "Test")] - r.loc[b, ("RMSE", "Test")],
                "dDirAcc Test": r.loc[a, ("DirAcc", "Test")] - r.loc[b, ("DirAcc", "Test")], "p DM Test": dm_test(te.target, PT[a], PT[b])[1]})
pd.DataFrame(cmp).round(5)

,sumber,dRMSE Val,dRMSE Test,dDirAcc Test,p DM Test
0,sent_lama,-0.00009,0.00001,-0.02841,0.37717
1,finbert,0.00017,0.00006,-0.01477,0.52807
2,deberta_ft,0.00007,0.00006,-0.01364,0.50134
3,bge_small,0.00017,0.00003,-0.02159,0.97429
4,bge_base,0.00008,-0.00016,0.02955,0.29773
5,deberta_base,-0.00011,-0.00002,0.05000,0.59732
